# Simple: Attribute Suppression with PAMOLA.CORE

**Goal**: Learn attribute suppression basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Remove sensitive columns using execute()
- Compare before/after column structures
- Understand column-level privacy protection

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/
        └── 01_attribute_suppression_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.suppression.attribute_op import AttributeSuppressionOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load from `examples/data_examples/sample.csv`
- Auto-creates 100 patient records if file missing
- Review dataset structure before suppression

**What you'll see:**
- Dataset summary (100 records, 10 columns)
- First 5 records with ALL columns (including sensitive)
- Column statistics (types, unique counts, date ranges)
- Sensitive fields: ssn, name, phone_number, email, insurance_id

**Note:** Auto-generated data includes realistic sensitive fields for suppression demonstration

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create realistic patient records with sensitive information
    np.random.seed(42)
    
    sample_data = pd.DataFrame({
        'patient_id': [f'P{i:04d}' for i in range(1, 101)],
        'name': [f'Patient {i}' for i in range(1, 101)],
        'ssn': [f'{np.random.randint(100, 999)}-{np.random.randint(10, 99)}-{np.random.randint(1000, 9999)}' for _ in range(100)],
        'date_of_birth': pd.date_range('1940-01-01', periods=100, freq='120D'),
        'phone_number': [f'555-{np.random.randint(100, 999)}-{np.random.randint(1000, 9999)}' for _ in range(100)],
        'email': [f'patient{i}@email.com' for i in range(1, 101)],
        'diagnosis': np.random.choice(['Diabetes', 'Hypertension', 'Asthma', 'Arthritis'], 100),
        'admission_date': pd.date_range('2024-01-01', periods=100, freq='3D'),
        'blood_type': np.random.choice(['A+', 'A-', 'B+', 'B-', 'O+', 'O-', 'AB+', 'AB-'], 100),
        'insurance_id': [f'INS{np.random.randint(100000, 999999)}' for _ in range(100)]
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)

print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:20s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:20s}): {df[col].min()} to {df[col].max()}")
    else:
        print(f"  {col:20s} ({dtype_str:20s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field_name**: Edit `create_config_kwargs()` function
   - Default: `"resume_id"`
   - Change to YOUR primary column (e.g., "ssn", "patient_id")
2. Run to validate and setup environment

**What you'll see:**
- Field validation (type, unique values, sample values)
- Task directory created (✓)
- TaskReporter, DataSource, progress tracker initialized (✓)

**Note:** field_name is PRIMARY column. Add more fields in Step 4's additional_fields parameter

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "resume_id"
    }

kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to suppress: '{field_name}'")
print(f"  Data type: {df[field_name].dtype}")
print(f"  Non-null values: {df[field_name].notna().sum()}")
print(f"  Unique values: {df[field_name].nunique()}")
print(f"  Sample values: {list(df[field_name].head(3))}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_suppression_001",
    task_type="attribute_suppression",
    description="Attribute suppression for sensitive patient information",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Suppressing {field_name} and additional fields",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- **CUSTOMIZE additional_fields**: Add more columns to suppress
- Run to execute suppression operation
- Monitor progress bar (6 steps: validate → identify → backup → remove → output → artifacts)

**Key parameters:**
- `additional_fields=['phone', 'email']`: Extra columns to remove
- `suppression_mode='REMOVE'`: Permanently removes columns
- `save_suppressed_schema=True`: Saves metadata about removed columns

**What you'll see:**
- Configuration summary (primary + additional fields, total count)
- Progress bar through 6 processing steps
- Completion status with artifact count (4-6 files)
- Success message (✅ Operation completed!)

**Note:** Execution takes ~1-3 seconds for 100 records. ⚠️ Suppressed columns are PERMANENTLY removed (irreversible)

In [ ]:
# Create operation with production-style configuration
operation = AttributeSuppressionOperation(
    # Core parameters
    field_name=field_name,
    
    # Suppression parameters
    additional_fields=['phone', 'email'],  # Additional columns to remove
    suppression_mode='REMOVE',                     # Must be 'REMOVE' for attribute suppression
    save_suppressed_schema=True,                   # Save metadata about suppressed columns
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Primary field:        {operation.field_name}")
print(f"  Additional fields:    {', '.join(operation.additional_fields)}")
print(f"  Total to suppress:    {1 + len(operation.additional_fields)} columns")
print(f"  Suppression mode:     {operation.suppression_mode}")
print(f"  Save schema:          {operation.save_suppressed_schema}")
print(f"  Visualizations:       {operation.generate_visualization}")
print(f"  Save output:          {operation.save_output}")
print(f"  Force recalc:         {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing Attribute Suppression Operation...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and compare processed data
- Review column structure changes

**What you'll see:**
- Artifacts list (CSV, JSON, PNG files)
- Sample records (first 10) with only remaining columns
- Column comparison (original vs suppressed vs remaining)
- Data width reduction (% of columns removed)
- Summary (record count unchanged, columns reduced)

**Note:** 100% of records preserved - suppression only removes columns, not rows. Reduction ≥30% = strong privacy, 15-30% = good balance, <15% = limited privacy

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records
    print("\n🔍 Sample Processed Records (First 10):")
    print(result_df.head(10))
    
    # Column comparison
    print("\n📋 Column Structure Comparison:")
    original_cols = set(df.columns)
    result_cols = set(result_df.columns)
    suppressed_cols = original_cols - result_cols
    remaining_cols = result_cols
    
    print(f"\n  Original columns ({len(original_cols)}):")
    print(f"    {', '.join(sorted(original_cols))}")
    
    print(f"\n  Suppressed columns ({len(suppressed_cols)}):")
    print(f"    {', '.join(sorted(suppressed_cols))}")
    
    print(f"\n  Remaining columns ({len(remaining_cols)}):")
    print(f"    {', '.join(sorted(remaining_cols))}")
    
    # Data width reduction
    reduction = (len(suppressed_cols) / len(original_cols)) * 100
    print(f"\n📉 Data Width Reduction:")
    print(f"  Columns removed:    {len(suppressed_cols)}")
    print(f"  Columns remaining:  {len(remaining_cols)}")
    print(f"  Reduction:          {reduction:.2f}%")
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:   {len(df)}")
    print(f"  Processed records:  {len(result_df)}")
    print(f"  Records unchanged:  {len(df) == len(result_df)}")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:15]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:35s}: {value:.4f}")
                else:
                    print(f"  {key:35s}: {value}")
            elif isinstance(value, list):
                print(f"  {key:35s}: {', '.join(map(str, value))}")
    
    print(f"\n💡 Attribute Suppression Insight:")
    print(f"   Sensitive columns (SSN, phone, email) have been completely removed from the dataset.")
    print(f"   This provides strong privacy protection by eliminating direct identifiers.")
    print(f"   The remaining {len(remaining_cols)} columns can still be used for analysis.")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, metrics/, visualizations/, config/)
- File count per directory (typically 1-3 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Anonymized CSV (columns removed)
├── metrics/          # Metrics + suppressed schema JSON
├── visualizations/   # Before/after comparison charts
└── config/           # Operation configuration
```

**Note:** Suppressed schema JSON in metrics/ contains metadata about removed columns for audit/compliance

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Focus on validation points in output

**What you'll see:**
1. **Metrics JSON**: Performance, suppression stats, column names removed
2. **Suppressed Schema**: Metadata per removed column (type, unique count, memory)
3. **Output Data**: Remaining columns, sample records, field distributions
4. **Visualizations**: Before/after comparison charts
5. **Privacy Impact**: Reduction %, protection level assessment

**Validate:**
- ✅ Record count unchanged (no data loss)
- ✅ Sensitive columns removed from output
- ✅ Reduction % matches expectations
- ✅ Analytical columns still present

**Note:** Only newest files shown. Multiple runs create versions - older artifacts excluded automatically

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# ============================================================================
# 1. METRICS JSON ANALYSIS (NEWEST)
# ============================================================================
metrics_files = list((task_dir / 'metrics').glob('*_metrics_*.json'))
if metrics_files:
    # 1) Exclude data_types inside IF
    filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

    if filtered:
        # Use only filtered files
        target_files = filtered
        print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
    else:
        # Fallback: nothing left after filtering → use all
        target_files = metrics_files
        print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

    # 2) Pick latest
    target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest_metrics_file = target_files[0]

    print(f"📄 Loading LATEST: {latest_metrics_file.name}")
    mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
    print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
    print("=" * 80)
    
    with open(latest_metrics_file) as f:
        data = json.load(f)
    
    # Extract metrics from nested structure
    metrics = data.get('metrics', data)
    
    # Display metadata if available
    if 'metadata' in data:
        print("\n📝 Metadata:")
        print(f"  Timestamp:   {data['metadata'].get('timestamp', 'N/A')}")
        print(f"  Operation:   {data['metadata'].get('operation', {}).get('class', 'N/A')}")
    
    # Display performance metrics
    if 'duration_seconds' in metrics:
        print("\n⚡ Performance Metrics:")
        print(f"  Duration:             {metrics.get('duration_seconds', 0):.2f} seconds")
        print(f"  Records processed:    {metrics.get('records_processed', 0):,}")
        print(f"  Processing rate:      {metrics.get('records_per_second', 0):,.2f} records/sec")
        print(f"  Chunk count:          {metrics.get('chunk_count', 0)}")
    
    # Display suppression metrics
    if 'operation_type' in metrics:
        print("\n🗑️  Suppression Metrics:")
        print(f"  Operation type:       {metrics.get('operation_type', 'N/A')}")
        print(f"  Columns suppressed:   {metrics.get('columns_suppressed', 0)}")
        if 'suppressed_column_names' in metrics:
            print(f"  Suppressed columns:   {', '.join(metrics.get('suppressed_column_names', []))}")
        print(f"  Data width reduction: {metrics.get('data_width_reduction', 0):.2f}%")
    
    # Display record statistics
    if 'total_records' in metrics:
        print("\n📈 Record Statistics:")
        total = metrics.get('total_records', 0)
        processed = metrics.get('processed_records', 0)
        filtered = metrics.get('filtered_records', 0)
        print(f"  Total records:        {total:,}")
        print(f"  Processed records:    {processed:,} ({metrics.get('processing_rate', 0):.2f}%)")
        print(f"  Filtered records:     {filtered:,}")
else:
    print("\n⚠️  No metrics files found")

# ============================================================================
# 2. SUPPRESSED SCHEMA JSON ANALYSIS (NEWEST)
# ============================================================================
schema_files = list((task_dir / 'metrics').glob('*_suppressed_columns_schema_*.json'))
if schema_files:
    schema_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest_schema = schema_files[0]
    print(f"\n\n📋 Suppressed Column Schema: {latest_schema.name}")
    print("=" * 80)
    
    with open(latest_schema) as f:
        schema = json.load(f)
    
    print(f"\n✓ Found metadata for {len(schema)} suppressed column(s)\n")
    
    for col_name, col_info in schema.items():
        print(f"\n{'─' * 80}")
        print(f"  Column: {col_name}")
        print(f"{'─' * 80}")
        
        if isinstance(col_info, dict):
            # Basic information
            if 'dtype' in col_info:
                print(f"    Data Type:        {col_info['dtype']}")
            if 'memory_usage' in col_info:
                mem_mb = col_info['memory_usage'] / (1024 * 1024)
                print(f"    Memory Usage:     {mem_mb:.2f} MB")
            
            # Count statistics
            if 'non_null_count' in col_info:
                print(f"    Non-null values:  {col_info['non_null_count']:,}")
            if 'null_count' in col_info:
                print(f"    Null values:      {col_info['null_count']:,}")
            if 'unique_count' in col_info:
                print(f"    Unique values:    {col_info['unique_count']:,}")
            
            # Numeric statistics (if available)
            numeric_keys = ['min', 'max', 'mean', 'std']
            has_numeric = any(k in col_info for k in numeric_keys)
            if has_numeric:
                print(f"\n    Numeric Statistics:")
                if 'min' in col_info:
                    print(f"      Min:    {col_info['min']:.4f}")
                if 'max' in col_info:
                    print(f"      Max:    {col_info['max']:.4f}")
                if 'mean' in col_info:
                    print(f"      Mean:   {col_info['mean']:.4f}")
                if 'std' in col_info:
                    print(f"      Std:    {col_info['std']:.4f}")
        else:
            print(f"    {col_info}")
    
    print(f"\n{'─' * 80}")
else:
    print("\n⚠️  No suppressed schema files found")

# ============================================================================
# 3. OUTPUT DATA COMPARISON (NEWEST)
# ============================================================================
print(f"\n\n📊 Output Data Analysis:")
print("=" * 80)

output_files = list((task_dir / 'output').glob('*.csv'))
if output_files:
    output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest_output = output_files[0]
    output_df = pd.read_csv(latest_output)
    
    print(f"\n✓ Loaded: {latest_output.name}")
    print(f"  Records: {len(output_df):,}")
    print(f"  Columns: {len(output_df.columns)}")
    print(f"  File size: {latest_output.stat().st_size / 1024:.1f} KB")
    
    print("\n📋 Column Structure:")
    print(f"\n  Remaining columns after suppression:")
    for i, col in enumerate(output_df.columns, 1):
        dtype = output_df[col].dtype
        nunique = output_df[col].nunique()
        print(f"    {i:2d}. {col:25s} ({str(dtype):15s}) - {nunique:,} unique values")
    
    print("\n📈 Data Sample (First 10 Records):")
    print(output_df.head(10).to_string(index=False))
    
    # Show distribution of key remaining fields
    if len(output_df.columns) > 0:
        print("\n📊 Sample Field Distributions:")
        
        # Analyze first 3 columns for demonstration
        for col in output_df.columns[:3]:
            print(f"\n{'─' * 80}")
            print(f"  Column: {col}")
            print(f"{'─' * 80}")
            
            if pd.api.types.is_string_dtype(df[col]):
                print(f"  Type: Categorical/String")
                print(f"  Top 5 values:")
                value_counts = output_df[col].value_counts().head(5)
                for val, count in value_counts.items():
                    pct = (count / len(output_df)) * 100
                    bar = '█' * int(pct / 2)
                    print(f"    {str(val):30s}: {count:5,} ({pct:5.1f}%) {bar}")
            elif pd.api.types.is_numeric_dtype(output_df[col]):
                print(f"  Type: Numeric")
                print(f"  Statistics:")
                print(f"    Count:  {output_df[col].count():,}")
                print(f"    Min:    {output_df[col].min()}")
                print(f"    Max:    {output_df[col].max()}")
                print(f"    Mean:   {output_df[col].mean():.2f}")
                print(f"    Median: {output_df[col].median():.2f}")
                print(f"    Std:    {output_df[col].std():.2f}")
            elif pd.api.types.is_datetime64_any_dtype(output_df[col]):
                print(f"  Type: Datetime")
                print(f"  Range:")
                print(f"    Earliest: {output_df[col].min()}")
                print(f"    Latest:   {output_df[col].max()}")
        
        print(f"\n{'─' * 80}")
else:
    print("\n⚠️  No output files found")

# ============================================================================
# 4. VISUALIZATIONS DISPLAY (NEWEST BATCH)
# ============================================================================
print(f"\n\n🎨 Generated Visualizations:")
print("=" * 80)

viz_files = list((task_dir / 'visualizations').glob('*.png'))
if viz_files:
    viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest_time = viz_files[0].stat().st_mtime
    # Group visualizations from the same batch (within 10 seconds)
    latest_batch = [f for f in viz_files if abs(f.stat().st_mtime - latest_time) < 10]
    
    print(f"\n✓ Found {len(latest_batch)} visualization(s) from latest batch")
    print(f"  Generated at: {datetime.fromtimestamp(latest_time).strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    for i, viz_file in enumerate(latest_batch, 1):
        # Create readable title from filename
        title = viz_file.stem.replace('_', ' ').replace('attributesuppressionoperation', '').strip()
        title = ' '.join(word.capitalize() for word in title.split())
        
        print(f"\n{'═' * 80}")
        print(f"📊 Visualization {i}/{len(latest_batch)}: {title}")
        print(f"{'═' * 80}\n")
        
        # Display the image
        display(Image(filename=str(viz_file)))
        
        # Show file metadata
        size_kb = viz_file.stat().st_size / 1024
        print(f"\n   📁 File: {viz_file.name}")
        print(f"   📏 Size: {size_kb:.1f} KB")
        
        if i < len(latest_batch):
            print("\n" + "─" * 80)
else:
    print("\n⚠️  No visualization files found")

# ============================================================================
# 5. PRIVACY IMPACT SUMMARY
# ============================================================================
print(f"\n\n🔒 Privacy Impact Summary:")
print("=" * 80)

if metrics_files and 'data_width_reduction' in metrics:
    reduction = metrics['data_width_reduction']
    cols_suppressed = metrics.get('columns_suppressed', 0)
    
    print(f"\n✓ Data Width Reduction: {reduction:.2f}%\n")
    
    # Calculate original columns
    original_cols = cols_suppressed + len(output_df.columns) if output_files else cols_suppressed
    
    print(f"  📊 Column Statistics:")
    print(f"     Original columns:  {original_cols}")
    print(f"     Remaining columns: {len(output_df.columns) if output_files else 'N/A'}")
    print(f"     Suppressed:        {cols_suppressed} columns")
    
    if 'suppressed_column_names' in metrics:
        print(f"\n  🗑️  Suppressed Column Names:")
        for col in metrics['suppressed_column_names']:
            print(f"     • {col}")
    
    # Privacy protection level assessment
    print(f"\n  🛡️  Privacy Protection Level:")
    if reduction >= 30:
        level = "HIGH"
        icon = "🟢"
        desc = "Excellent privacy protection - significant reduction in identifying information"
    elif reduction >= 15:
        level = "MEDIUM"
        icon = "🟡"
        desc = "Good privacy protection - moderate reduction in sensitive attributes"
    else:
        level = "LOW"
        icon = "🟠"
        desc = "Basic privacy protection - limited attribute removal"
    
    print(f"     {icon} {level} - {reduction:.0f}% attribute reduction")
    print(f"     {desc}")
    
    # Key insights
    print(f"\n  💡 Key Insights:")
    print(f"     ✓ All direct identifiers have been permanently removed from the dataset")
    print(f"     ✓ Remaining {len(output_df.columns) if output_files else 'N/A'} attributes maintain full record integrity")
    print(f"     ✓ Re-identification risk significantly reduced through attribute elimination")
    print(f"     ✓ Dataset remains fully usable for statistical analysis and reporting")
    print(f"     ✓ Suppressed column metadata preserved for audit and compliance purposes")
    
    # Data utility assessment
    if output_files:
        utility_pct = (len(output_df.columns) / original_cols) * 100
        print(f"\n  📈 Data Utility Retention:")
        print(f"     {utility_pct:.1f}% of original attributes retained")
        if utility_pct >= 70:
            print(f"     ✓ High utility - most analytical capabilities preserved")
        elif utility_pct >= 50:
            print(f"     ✓ Moderate utility - core analytical capabilities maintained")
        else:
            print(f"     ⚠️  Reduced utility - consider balancing privacy and usability needs")
else:
    print("\n⚠️  Privacy metrics not available")

# ============================================================================
# 6. BEFORE/AFTER COMPARISON
# ============================================================================
print(f"\n\n📊 Before/After Comparison:")
print("=" * 80)

if output_files:
    print(f"\n  Original Dataset:")
    print(f"     Records: {len(df):,}")
    print(f"     Columns: {len(df.columns)}")
    print(f"     Size:    {df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")
    
    print(f"\n  Processed Dataset:")
    print(f"     Records: {len(output_df):,}")
    print(f"     Columns: {len(output_df.columns)}")
    print(f"     Size:    {output_df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")
    
    print(f"\n  Changes:")
    record_change = len(output_df) - len(df)
    column_change = len(output_df.columns) - len(df.columns)
    size_original = df.memory_usage(deep=True).sum() / (1024*1024)
    size_processed = output_df.memory_usage(deep=True).sum() / (1024*1024)
    size_change = size_processed - size_original
    
    print(f"     Records: {record_change:+,} ({'no change' if record_change == 0 else 'changed'})")
    print(f"     Columns: {column_change:+} ({abs(column_change)} removed)")
    print(f"     Size:    {size_change:+.2f} MB ({(size_change/size_original)*100:+.1f}%)")

# ============================================================================
# COMPLETION MESSAGE
# ============================================================================
print("\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)


## 🎯 Summary

**What you learned:**

✅ **Load data**: Load patient data from examples/data_examples/patient_records.csv  
✅ **Setup environment**: Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Configure attribute suppression to remove multiple sensitive columns  
✅ **Execute**: Execute using operation.execute() method  
✅ **Analyze results**: Analyze column removal results and privacy impact metrics  
✅ **Review artifacts**: Review comprehensive artifacts (CSV outputs, metrics JSON, visualizations)  

**Key takeaways:**
- Attribute suppression completely removes sensitive columns (SSN, phone, email) from the dataset
- Provides strong privacy protection by permanently eliminating direct identifiers
- Data remains fully usable with remaining demographic and clinical columns
- Metadata about suppressed columns is preserved for audit and compliance purposes
- ~30% data width reduction achieved while maintaining 100% record integrity
- Simple operation with powerful privacy impact - just specify field names to suppress
```

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_attribute_suppression_advanced.ipynb`** includes:
- Conditional suppression based on field values (remove SSN only for age < 18)
- Risk-based suppression using k-anonymity thresholds
- Multi-condition filtering with AND/OR logic
- Batch processing with chunk_size and memory optimization
- Cache management for large datasets (use_cache, force_recalculation)
- Advanced visualizations with custom backends and themes
- Encryption and security features for sensitive outputs
- Dask integration for distributed processing (10M+ records)

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)